# 02 — Data Cleaning & Preprocessing
**FlightIQ — AI Travel Price Intelligence**

MIC AIML Department Recruitment Challenge  
Track: Data Science & Visualization — AI Travel Analyst

---
**Goal**: Clean the raw dataset, handle mixed formats, impute missing values intelligently, and produce a model-ready CSV. No ML, no EDA, no Streamlit.

**Design principles**:
- No target leakage: `Price` is never used to construct input features.
- Preserve as much valid data as possible.
- All raw string traceability columns are retained alongside clean engineered columns.
- Deterministic: same input → same output, always.

In [1]:
import sys
import os
import re
import pandas as pd
import numpy as np

sys.path.insert(0, os.path.abspath('../src'))
from data_loader import load_raw_data
from data_preprocessing import (
    strip_whitespace, clean_price, clean_duration, clean_total_stops,
    clean_numeric_features, clean_airline, clean_city_columns,
    clean_categorical_columns, clean_datetime_columns,
    handle_outliers, drop_duplicates, run_cleaning_pipeline
)

pd.set_option('display.max_columns', None)
pd.set_option('display.width', 200)
print('Imports OK')

Imports OK


## 1. Load Raw Data

In [2]:
df_raw = load_raw_data()
ORIGINAL_SHAPE = df_raw.shape
ORIGINAL_MISSING = int(df_raw.isnull().sum().sum())
ORIGINAL_DUPES = int(df_raw.duplicated().sum())

print(f'Raw shape   : {ORIGINAL_SHAPE}')
print(f'Raw missing : {ORIGINAL_MISSING:,}')
print(f'Raw dupes   : {ORIGINAL_DUPES:,}')

Raw shape   : (100000, 18)
Raw missing : 82,966
Raw dupes   : 1,961


## 2. Dataset Inspection Before Cleaning

In [3]:
print('=== Raw dtypes ===')
print(df_raw.dtypes)
print('\n=== Missing per column ===')
miss = df_raw.isnull().sum()
print(miss[miss > 0])

=== Raw dtypes ===
Flight_ID                object
Airline                  object
Source                   object
Destination              object
Departure_Date           object
Departure_Time           object
Arrival_Time             object
Duration                 object
Total_Stops              object
Distance_km              object
Travel_Class             object
Days_Before_Departure    object
Season                   object
Weekday                  object
Aircraft_Type            object
Booking_Channel          object
Passenger_Count          object
Price                    object
dtype: object

=== Missing per column ===
Airline                  4880
Source                   4915
Destination              4881
Departure_Date           4927
Departure_Time           4868
Arrival_Time             4789
Duration                 5042
Total_Stops              4905
Distance_km              4923
Travel_Class             4863
Days_Before_Departure    4784
Season                   4776
Wee

In [4]:
print('=== Duration unique formats (sample) ===')
print(df_raw['Duration'].dropna().unique()[:20].tolist())

print('\n=== Total_Stops unique values ===')
print(df_raw['Total_Stops'].dropna().unique().tolist())

print('\n=== Price sample ===')
print(df_raw['Price'].dropna().unique()[:15].tolist())

print('\n=== Passenger_Count unique ===')
print(sorted(df_raw['Passenger_Count'].dropna().astype(str).unique().tolist()))

print('\n=== Airline unique (shows case issue) ===')
print(df_raw['Airline'].dropna().unique()[:10].tolist())

=== Duration unique formats (sample) ===
['1.67', '0h 45m', '14.80', '3h 11m', '13.93', '16h 47m', '1h 28m', '5h 27m', '17h 04m', '3h 20m', '177 min', '0h 59m', '2h 02m', '155 min', '22h 40m', '18h 45m', '6h 04m', '1.98', '2h 38m', '4.38']

=== Total_Stops unique values ===
['0', 'non-stop', '1 stop', '1', '2 stops', '2']

=== Price sample ===
['5181.56', '2000', '174762.69', '2846.09', '145331.64', '190059.64', '6458.88', '28599.41', '147645.15', '24660.74', '36225.75', '15000', '200000', '20785.21', '32822.58']

=== Passenger_Count unique ===
['1', '2', '3', '4', '5', '6', 'five', 'four', 'one', 'six', 'three', 'two']

=== Airline unique (shows case issue) ===
['Indigo', 'AirAsia India', 'Qatar Airways', 'British Airways', 'GoFirst', 'Singapore Airlines', 'AIR INDIA', 'Emirates', 'Etihad Airways', 'Thai Airways']


## 3. Duration Format Analysis

Three formats detected in the actual dataset:

| Format | Example | Interpretation |
|--------|---------|----------------|
| Decimal float | `"1.67"` | Hours (verified: 0.75h = 45min = matches `"0h 45m"`) |
| Hours+minutes | `"0h 45m"`, `"1h 28m"` | Direct hours + minutes |
| Minutes suffix | `"177 min"`, `"900 min"` | Already in minutes |

In [5]:
# Verify: 0.75 hours * 60 = 45 minutes = "0h 45m" ✓
print(f'0.75 * 60 = {0.75 * 60} min  →  matches "0h 45m"')
print(f'1.67 * 60 = {1.67 * 60:.1f} min  →  reasonable domestic flight')
print(f'14.80 * 60 = {14.80 * 60:.0f} min  →  reasonable long-haul flight')

# Count each format
dur = df_raw['Duration'].dropna()
fmt_float = dur[dur.str.match(r'^\d+(\.\d+)?$')].count()
fmt_hm = dur[dur.str.match(r'^\d+h\s*\d+m$')].count()
fmt_min = dur[dur.str.contains('min', na=False)].count()

print(f'\nDuration format counts:')
print(f'  Decimal hours : {fmt_float:,}')
print(f'  "Xh Ym"       : {fmt_hm:,}')
print(f'  "X min"       : {fmt_min:,}')

0.75 * 60 = 45.0 min  →  matches "0h 45m"
1.67 * 60 = 100.2 min  →  reasonable domestic flight
14.80 * 60 = 888 min  →  reasonable long-haul flight

Duration format counts:
  Decimal hours : 12,568
  "Xh Ym"       : 69,962
  "X min"       : 12,428


## 4. Run Full Cleaning Pipeline

In [6]:
df_clean, orig_shape, orig_missing = run_cleaning_pipeline()

FlightIQ — Data Cleaning Pipeline



Raw shape     : (100000, 18)
Raw missing   : 82,966


Raw dupes     : 1,961

--- Step 1: Strip whitespace ---


--- Step 2: Clean Price (target) ---


  [Price] Dropped 5053 rows (null/invalid/non-positive Price).
--- Step 3: Clean Duration ---


...

  [Destination] Canonicalized. Imputed 4644 missing.
--- Step 8: Standardize other categoricals ---
  [Travel_Class] Standardized. Imputed 4626 missing with mode ('Economy').
  [Season] Standardized. Imputed 4496 missing with mode ('Monsoon').


  [Weekday] Standardized. Imputed 4494 missing with mode ('Sunday').
  [Aircraft_Type] Standardized. Imputed 4599 missing with mode ('Airbus A320').
  [Booking_Channel] Standardized. Imputed 4707 missing with mode ('Website').
--- Step 9: Parse date/time columns ---
  [Departure_Month] Extracted. Imputed 4678 missing.
  [Departure_DayOfYear] Extracted. Imputed 4678 missing.
  [Departure_DayOfWeek_Num] Extracted. Imputed 4678 missing.


  [Departure_Time_Minutes] Converted. Imputed 4632 missing.


  [Arrival_Time_Minutes] Converted. Imputed 4556 missing.
  [Weekday_Num] Mapped. Imputed 0 missing.
--- Step 10: Remove outliers ---
  [Outliers] Removed 0 clearly invalid records.
--- Step 11: Drop duplicates ---
  [Duplicates] Removed 1864 exact duplicate rows.

Cleaned shape   : (93083, 27)
Cleaned missing : 27,435



Saved → /Users/jyotish/Documents/FlightIQ/data/processed/cleaned_flight_data.csv


## 5. Post-Cleaning Verification

In [7]:
print('=== Cleaned dtypes ===')
print(df_clean.dtypes)
print(f'\nTotal missing after cleaning: {df_clean.isnull().sum().sum():,}')
print('\nRemaining missing (raw traceability columns only):')
m = df_clean.isnull().sum()
print(m[m > 0])

=== Cleaned dtypes ===
Flight_ID                          object
Airline                            object
Source                             object
Destination                        object
Departure_Date                     object
Departure_Time                     object
Arrival_Time                       object
Duration                           object
Total_Stops                        object
Distance_km                       float64
Travel_Class                       object
Days_Before_Departure             float64
Season                             object
Weekday                            object
Aircraft_Type                      object
Booking_Channel                    object
Passenger_Count                   float64
Price                             float64
Duration_Minutes                  float64
Total_Stops_Numeric               float64
Departure_Date_Parsed      datetime64[ns]
Departure_Month                   float64
Departure_DayOfYear               float64
Departure_D

In [8]:
print('=== Duration_Minutes stats ===')
print(df_clean['Duration_Minutes'].describe())

print('\n=== Total_Stops_Numeric value counts ===')
print(df_clean['Total_Stops_Numeric'].value_counts().sort_index())

print('\n=== Price stats ===')
print(df_clean['Price'].describe())

=== Duration_Minutes stats ===
count    93083.000000
mean       481.777386
std        350.875969
min         45.000000
25%        205.000000
50%        354.000000
75%        713.000000
max       1647.000000
Name: Duration_Minutes, dtype: float64

=== Total_Stops_Numeric value counts ===
Total_Stops_Numeric
0.0    35646
1.0    45900
2.0    11537
Name: count, dtype: int64

=== Price stats ===
count     93083.000000
mean      72990.281711
std       77721.801978
min         152.130000
25%       13258.570000
50%       49100.130000
75%      113660.475000
max      999306.030000
Name: Price, dtype: float64


In [9]:
print('=== Airline value counts (after normalization) ===')
print(df_clean['Airline'].value_counts())

print('\n=== Source unique cities ===')
print(sorted(df_clean['Source'].unique().tolist()))

print('\n=== Passenger_Count value counts ===')
print(df_clean['Passenger_Count'].value_counts().sort_index())

=== Airline value counts (after normalization) ===
Airline
Singapore Airlines    12852
Lufthansa              8207
Etihad Airways         8206
Emirates               8161
Thai Airways           8155
Qatar Airways          8105
British Airways        8007
Vistara                5966
Air India              5876
Airasia India          4919
Indigo                 4914
Spicejet               4859
Gofirst                4856
Name: count, dtype: int64

=== Source unique cities ===
['Ahmedabad', 'Bangalore', 'Bangkok', 'Chennai', 'Delhi', 'Doha', 'Dubai', 'Frankfurt', 'Goa', 'Hyderabad', 'Jaipur', 'Kolkata', 'London', 'Mumbai', 'New York', 'Pune', 'Singapore', 'Sydney']

=== Passenger_Count value counts ===
Passenger_Count
1.0    41707
2.0    26214
3.0    11649
4.0     7194
5.0     3665
6.0     2654
Name: count, dtype: int64


In [10]:
# All engineered columns should have zero missing
engineered = [
    'Duration_Minutes', 'Total_Stops_Numeric', 'Distance_km',
    'Days_Before_Departure', 'Passenger_Count', 'Departure_Month',
    'Departure_DayOfYear', 'Departure_DayOfWeek_Num',
    'Departure_Time_Minutes', 'Arrival_Time_Minutes', 'Weekday_Num',
    'Airline', 'Source', 'Destination', 'Travel_Class', 'Season',
    'Weekday', 'Aircraft_Type', 'Booking_Channel', 'Price'
]
print('=== Engineered / cleaned columns — missing count ===')
for c in engineered:
    miss = df_clean[c].isna().sum()
    status = '✓' if miss == 0 else f'✗  {miss} missing'
    print(f'  {c:35s}: {status}')

=== Engineered / cleaned columns — missing count ===
  Duration_Minutes                   : ✓
  Total_Stops_Numeric                : ✓
  Distance_km                        : ✓
  Days_Before_Departure              : ✓
  Passenger_Count                    : ✓
  Departure_Month                    : ✓
  Departure_DayOfYear                : ✓
  Departure_DayOfWeek_Num            : ✓
  Departure_Time_Minutes             : ✓
  Arrival_Time_Minutes               : ✓
  Weekday_Num                        : ✓
  Airline                            : ✓
  Source                             : ✓
  Destination                        : ✓
  Travel_Class                       : ✓
  Season                             : ✓
  Weekday                            : ✓
  Aircraft_Type                      : ✓
  Booking_Channel                    : ✓
  Price                              : ✓


## 6. Check No Leakage

In [11]:
# Verify Price was NOT used to construct any feature
feature_cols = [c for c in df_clean.columns if c != 'Price']
print('Feature columns (Price excluded):')
print(feature_cols)
print('\nTarget column: Price')
print(f'  dtype  : {df_clean["Price"].dtype}')
print(f'  min    : {df_clean["Price"].min():,.2f}')
print(f'  max    : {df_clean["Price"].max():,.2f}')
print(f'  median : {df_clean["Price"].median():,.2f}')

Feature columns (Price excluded):
['Flight_ID', 'Airline', 'Source', 'Destination', 'Departure_Date', 'Departure_Time', 'Arrival_Time', 'Duration', 'Total_Stops', 'Distance_km', 'Travel_Class', 'Days_Before_Departure', 'Season', 'Weekday', 'Aircraft_Type', 'Booking_Channel', 'Passenger_Count', 'Duration_Minutes', 'Total_Stops_Numeric', 'Departure_Date_Parsed', 'Departure_Month', 'Departure_DayOfYear', 'Departure_DayOfWeek_Num', 'Departure_Time_Minutes', 'Arrival_Time_Minutes', 'Weekday_Num']

Target column: Price
  dtype  : float64
  min    : 152.13
  max    : 999,306.03
  median : 49,100.13


## 7. Final Summary

In [12]:
print('=' * 60)
print('PART 1 — DATA CLEANING SUMMARY')
print('=' * 60)
print(f'Original shape      : {ORIGINAL_SHAPE}')
print(f'Cleaned shape       : {df_clean.shape}')
print(f'Target column       : Price')
print(f'Missing before      : {ORIGINAL_MISSING:,}')
print(f'Missing after       : {df_clean.isnull().sum().sum():,}')
print(f'(Remaining missing are raw traceability columns only)')
print(f'Rows dropped (Price): 5,053')
print(f'Duplicates removed  : 1,864')
print(f'New columns added   : Duration_Minutes, Total_Stops_Numeric,')
print(f'                      Departure_Month, Departure_DayOfYear,')
print(f'                      Departure_DayOfWeek_Num, Departure_Time_Minutes,')
print(f'                      Arrival_Time_Minutes, Weekday_Num, Departure_Date_Parsed')
print(f'Output saved        : data/processed/cleaned_flight_data.csv')
print('=' * 60)

PART 1 — DATA CLEANING SUMMARY
Original shape      : (100000, 18)
Cleaned shape       : (93083, 27)
Target column       : Price
Missing before      : 82,966
Missing after       : 27,435
(Remaining missing are raw traceability columns only)
Rows dropped (Price): 5,053
Duplicates removed  : 1,864
New columns added   : Duration_Minutes, Total_Stops_Numeric,
                      Departure_Month, Departure_DayOfYear,
                      Departure_DayOfWeek_Num, Departure_Time_Minutes,
                      Arrival_Time_Minutes, Weekday_Num, Departure_Date_Parsed
Output saved        : data/processed/cleaned_flight_data.csv
